In [1]:
import pandas as pd

In [4]:
def preprocess_air_quality(file_path):
    # Load the dataset
    df = pd.read_csv(file_path)

    # Clean column names
    df.columns = df.columns.str.strip()

    # Convert date column to datetime format
    df["date"] = pd.to_datetime(df["date"], format="%d-%m-%Y")

    # Convert pollutant columns to numeric
    pollutants = ["pm25", "pm10", "o3", "no2", "so2", "co"]
    df[pollutants] = df[pollutants].apply(pd.to_numeric, errors="coerce")

    # Handle missing values using forward fill
    df.fillna(method='ffill', inplace=True)

    # Define AQI breakpoints for each pollutant based on the US EPA standard
    aqi_breakpoints = {
        'pm25': [
            (0.0, 12.0, 0, 50),
            (12.1, 35.4, 51, 100),
            (35.5, 55.4, 101, 150),
            (55.5, 150.4, 151, 200),
            (150.5, 250.4, 201, 300),
            (250.5, 350.4, 301, 400),
            (350.5, 500.4, 401, 500)
        ],
        'pm10': [
            (0, 54, 0, 50),
            (55, 154, 51, 100),
            (155, 254, 101, 150),
            (255, 354, 151, 200),
            (355, 424, 201, 300),
            (425, 504, 301, 400),
            (505, 604, 401, 500)
        ],
        'o3': [
            (0, 54, 0, 50),
            (55, 70, 51, 100),
            (71, 85, 101, 150),
            (86, 105, 151, 200),
            (106, 200, 201, 300),
            (201, 300, 301, 400),
            (301, 500, 401, 500)
        ],
        'no2': [
            (0, 53, 0, 50),
            (54, 100, 51, 100),
            (101, 360, 101, 150),
            (361, 649, 151, 200),
            (650, 1249, 201, 300),
            (1250, 1649, 301, 400),
            (1650, 2049, 401, 500)
        ],
        'so2': [
            (0, 35, 0, 50),
            (36, 75, 51, 100),
            (76, 185, 101, 150),
            (186, 304, 151, 200),
            (305, 604, 201, 300),
            (605, 804, 301, 400),
            (805, 1004, 401, 500)
        ],
        'co': [
            (0.0, 4.4, 0, 50),
            (4.5, 9.4, 51, 100),
            (9.5, 12.4, 101, 150),
            (12.5, 15.4, 151, 200),
            (15.5, 30.4, 201, 300),
            (30.5, 40.4, 301, 400),
            (40.5, 50.4, 401, 500)
        ]
    }

    # Function to calculate AQI for a given pollutant
    def calculate_pollutant_aqi(concentration, breakpoints):
        for bp_low, bp_high, aqi_low, aqi_high in breakpoints:
            if bp_low <= concentration <= bp_high:
                # Linear interpolation
                return ((concentration - bp_low) / (bp_high - bp_low)) * (aqi_high - aqi_low) + aqi_low
        return None

    # Function to calculate the overall AQI
    def calculate_aqi(row):
        aqi_values = []
        for pollutant, breakpoints in aqi_breakpoints.items():
            concentration = row[pollutant]
            if pd.notna(concentration):
                aqi = calculate_pollutant_aqi(concentration, breakpoints)
                if aqi is not None:
                    aqi_values.append(aqi)
        return max(aqi_values) if aqi_values else None

    # Calculate AQI for each row
    df['aqi'] = df.apply(calculate_aqi, axis=1)

    # Reorder columns to have 'date' first
    df = df[['date', 'pm25', 'pm10', 'o3', 'no2', 'so2', 'co', 'aqi']]

    return df


In [2]:
# File path
file_path = "C:/Users/misty/OneDrive/Desktop/AirVision AI Delhi/data/raw_delhi_air_quality_data.csv"

In [5]:
# Preprocess the data
df_cleaned = preprocess_air_quality(file_path)

In [6]:
# Display the cleaned dataset
df_cleaned.head()

,date,pm25,pm10,o3,no2,so2,co,aqi
0,2025-02-01,325.0,166.0,35.0,24.0,2.0,10.0,374.828829
1,2025-02-02,244.0,136.0,66.0,26.0,3.0,12.0,293.657658
2,2025-02-03,191.0,224.0,32.0,36.0,4.0,14.0,241.135135
3,2025-02-04,223.0,126.0,64.0,20.0,2.0,11.0,272.846847
4,2025-02-05,191.0,130.0,49.0,22.0,5.0,12.0,241.135135


In [7]:
df_cleaned.tail()

,date,pm25,pm10,o3,no2,so2,co,aqi
2548,2018-06-02,220.0,113.0,29.0,24.0,18.0,7.0,269.873874
2549,2018-01-18,220.0,515.0,29.0,24.0,18.0,7.0,411.000000
2550,2018-01-22,220.0,317.0,29.0,24.0,18.0,7.0,269.873874
2551,2018-03-18,220.0,148.0,27.0,21.0,9.0,7.0,269.873874
2552,2021-03-31,220.0,148.0,13.0,14.0,2.0,6.0,269.873874


In [8]:
df_cleaned.isnull().sum()

date    0
pm25    0
pm10    0
o3      0
no2     0
so2     0
co      0
aqi     0
dtype: int64

In [9]:
# Save cleaned dataset to CSV
df_cleaned.to_csv("cleaned_air_quality_data_delhi.csv", index=False)

print("Cleaned dataset saved successfully in 'cleaned_air_quality_data_delhi.csv'")


Cleaned dataset saved successfully in 'cleaned_air_quality_data_delhi.csv'


In [11]:
# Define the path to save the cleaned dataset
output_path = "C:/Users/misty/OneDrive/Desktop/AirVision AI Delhi/data/cleaned_delhi_air_quality_data.csv"

# Save cleaned dataset to CSV in the 'data' folder
df_cleaned.to_csv(output_path, index=False)

print(f"Cleaned dataset saved successfully in '{output_path}'")


Cleaned dataset saved successfully in 'C:/Users/misty/OneDrive/Desktop/AirVision AI Delhi/data/cleaned_delhi_air_quality_data.csv'


In [12]:
# Display the cleaned dataset
df_cleaned.head(20)

,date,pm25,pm10,o3,no2,so2,co,aqi
0,2025-02-01,325.0,166.0,35.0,24.0,2.0,10.0,374.828829
1,2025-02-02,244.0,136.0,66.0,26.0,3.0,12.0,293.657658
2,2025-02-03,191.0,224.0,32.0,36.0,4.0,14.0,241.135135
3,2025-02-04,223.0,126.0,64.0,20.0,2.0,11.0,272.846847
4,2025-02-05,191.0,130.0,49.0,22.0,5.0,12.0,241.135135
5,2025-02-06,172.0,133.0,56.0,22.0,5.0,13.0,222.306306
6,2025-02-07,152.0,156.0,43.0,42.0,7.0,12.0,202.486486
7,2025-02-08,173.0,192.0,48.0,65.0,11.0,10.0,223.297297
8,2025-02-09,193.0,293.0,48.0,75.0,7.0,9.0,243.117117
9,2025-02-10,231.0,334.0,60.0,71.0,7.0,14.0,280.774775
